#MVP

ЧТобы посчитать TF-IDF метрику, нам нужно вытащить из датасета посты. Также, нужно очистить данные от выбросов, поэтому из исходного data.json сделаем две таблицы: users_df -- очищенная таблица с пользователями, и posts_df -- посты этих пользователей.

In [6]:
#Загрузка и чистка данных
import json
import pandas as pd

with open('data.json', 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

users_list =[]
for u in raw_data:
    # фильтруем пустые страницы
    if u['friends_num'] > 0 and u['self_posts_num'] > 0:
        users_list.append({
            'user_id': u['id'],
            'city': u['city'],
            'age': u['age'],
            'gender': u['gender'],
            'friends_num': u['friends_num'],
            'groups_num': u['groups_num'],
            'self_posts_num': u['self_posts_num'],
            'reposts_num': u.get('reposts_num', 0)
        })

users_df = pd.DataFrame(users_list)

#убираем выбросы(блогеры с болоьшим кол-вом друзей и т.п.) и берем 95-й перцентиль
q95 = users_df['friends_num'].quantile(0.95)
users_df = users_df[users_df['friends_num'] <= q95]

#все посты в отдельную таблицу
posts_list =[]
for u in raw_data:
    if u['id'] in users_df['user_id'].values:
        for post_id, post_data in u['posts'].items():
            text = post_data['text'].strip()
            if len(text) > 10:
                posts_list.append({
                    'user_id': u['id'],
                    'post_id': post_id,
                    'text': text
                })

posts_df = pd.DataFrame(posts_list)

print(f"Осталось пользователей: {len(users_df)}")
print(f"Собрано постов для ML: {len(posts_df)}")

Осталось пользователей: 310
Собрано постов для ML: 10879


Для обучения ML модели возьмем датасет k1tub/sentiment_dataset, представляющий собой тексты из разных источников(новости, отзывы, посты в соц.сетях), размеченных по тональности. В датасете метка 0 означает нейтрально, 1 -- это позитив и 2 -- это негатив.

#MVP: TF-IDF и Логистическая регрессия

In [7]:
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import pandas as pd

dataset = load_dataset("k1tub/sentiment_dataset")

df = dataset['train'].to_pandas()

#оставим только позитивные и негативные, без нейтральных
df = df[df['label'].isin([1, 2])]

#тепрь 1 - хорошо, 0 - плохо
df['label'] = df['label'].replace(2, 0)


df_train = df.sample(100000, random_state=42)

train_texts = df_train['text'].tolist()
train_labels = df_train['label'].tolist()

print("Векторизация (то есть TF-IDF)")
vectorizer = TfidfVectorizer(max_features=10000)
X_train = vectorizer.fit_transform(train_texts)

print("Обучаем логистическую регрессию")
model = LogisticRegression(max_iter=1000)
model.fit(X_train, train_labels)



README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/96.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/290458 [00:00<?, ? examples/s]

Векторизация (то есть TF-IDF)
Обучаем логистическую регрессию


LogisticRegression(max_iter=1000)

Теперь у нас есть обученная model и vectorizer. Берем посыт (posts_df), переводим их в векторы и просим модель угадать, позитив это или негатив.

In [8]:
X_vk = vectorizer.transform(posts_df['text']) #посты из вк в векторы

posts_df['sentiment'] = model.predict(X_vk)

#группируем по user_id и считаем среднее (это доля позитивных постов)
sentiment_scores = posts_df.groupby('user_id')['sentiment'].mean().reset_index()
sentiment_scores.rename(columns={'sentiment': 'positive_posts_percent'}, inplace=True)

users_df = users_df.merge(sentiment_scores, on='user_id', how='left')

#если у кого-то все посты отфильтровались (например, были слишком короткие), ставим им 0.5 (т.е. нейтрально)
users_df['positive_posts_percent'] = users_df['positive_posts_percent'].fillna(0.5)

print(users_df[['user_id', 'self_posts_num', 'positive_posts_percent']].head())

     user_id  self_posts_num  positive_posts_percent
0  230173495              11                1.000000
1   33545305              58                0.760000
2  104916144              98                0.948980
3   42020745              97                0.808081
4  641099221              91                0.862069


Теперь соберем индекс счастья по формуле

In [9]:
from sklearn.preprocessing import MinMaxScaler


#добавим S_activity из ТЗ. (это общая активность по постам), т.е. свои посты и репосты
users_df['total_posts_activity'] = users_df['self_posts_num'] + users_df['reposts_num']
users_df = users_df[users_df['friends_num'] < 5000]

#нормализуем S_activity
scaler = MinMaxScaler()
users_df['activity_norm'] = scaler.fit_transform(users_df[['total_posts_activity']])
users_df['friends_norm'] = scaler.fit_transform(users_df[['friends_num']])


#для mvp задаем веса для всех трех компонентов вручную
alpha_1 = 0.3  #Вес социальных связей (S_network)
alpha_2 = 0.2  #Вес активности (S_activity)
alpha_3 = 0.5  #Вес позитивных эмоций (S_sentiment)

users_df['happiness_index'] = (users_df['friends_norm'] * alpha_1) + \
                              (users_df['activity_norm'] * alpha_2) + \
                              (users_df['positive_posts_percent'] * alpha_3)

users_df['happiness_index'] = users_df['happiness_index'].round(3)


print(users_df[['user_id', 'friends_num', 'friends_norm',
                'total_posts_activity', 'activity_norm',
                'positive_posts_percent', 'happiness_index']].sort_values(by='happiness_index', ascending=False).head(10))


       user_id  friends_num  friends_norm  total_posts_activity  \
280   29154851         3710      0.750962                   100   
76   137761692         4703      0.952015                   100   
263  135033512         4327      0.875886                   100   
140  338648691         4172      0.844503                   100   
10     1324639         4251      0.860498                   100   
112  374993176         4006      0.810893                   100   
126   55392984         3500      0.708443                   100   
89   311287157         3268      0.661470                   100   
9    766980427         2763      0.559223                   100   
189   48315616         4074      0.824661                   100   

     activity_norm  positive_posts_percent  happiness_index  
280            1.0                0.929412            0.890  
76             1.0                0.795699            0.883  
263            1.0                0.833333            0.879  
140           

(по 100 постов, так как рассматривали 100 последних постов(ограничение вк апи)), аналогично 5000 друзей это лимит вк апи. Из-за линейной нормализации MinMaxScaler такие люди получили максимальный балл активности. Это значит, что наша текущая линейная метрика слишком чувствительна к выбросам и популярности аккаунта.

upd: добавила


users_df = users_df[users_df['friends_num'] < 5000]



чтобы убрать людей, которые обрезались из-за лимитов. (скорее всего, это были магазины, блогеры и тому подобное)

#Основная часть проекта
Внедрение DL

In [ ]:
from transformers import pipeline
import pandas as pd
from tqdm import tqdm

sentiment_dl = pipeline("sentiment-analysis", model="blanchefort/rubert-base-cased-sentiment", truncation=True, max_length=512)

texts_list = posts_df['text'].tolist()

dl_sentiments = []
#tqdm, так как нейросеть думает чуть дольше
for text in tqdm(texts_list, desc="Анализ тональности (DL)"):
    try:
        result = sentiment_dl(text)[0]
        label = result['label']

        #позитив=1, негатив=0, нейтрально=0.5
        if label == 'POSITIVE':
            dl_sentiments.append(1.0)
        elif label == 'NEGATIVE':
            dl_sentiments.append(0.0)
        else:
            dl_sentiments.append(0.5)
    except Exception as e:
        dl_sentiments.append(0.5)

posts_df['dl_sentiment'] = dl_sentiments

print("\n Считаем процент позитива для каждого пользователя...")
dl_sentiment_scores = posts_df.groupby('user_id')['dl_sentiment'].mean().reset_index()
dl_sentiment_scores.rename(columns={'dl_sentiment': 'dl_positive_percent'}, inplace=True)

if 'dl_positive_percent' in users_df.columns:
    users_df.drop(columns=['dl_positive_percent'], inplace=True)

users_df = users_df.merge(dl_sentiment_scores, on='user_id', how='left')
users_df['dl_positive_percent'] = users_df['dl_positive_percent'].fillna(0.5)

print("\nСравнение двух подходов(ML vs DL):")
print(users_df[['user_id', 'positive_posts_percent', 'dl_positive_percent']].head(10))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: blanchefort/rubert-base-cased-sentiment
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Анализ тональности (DL):   4%|▍         | 476/10879 [09:49<3:01:05,  1.04s/it]

#Анализ тематики групп
Из диздока: "Учитывается штрафной коэффициент за наличие токсичных/депрессивных сообществ. Описание сообществ для определения их тематики (хобби, образование, негатив)".

 Для этого воспользуемся Keyword Extraction (поиск по ключевым словам) в описаниях и названиях групп.

In [ ]:
#маркеры?? тут что-то нужно придумать еще
toxic_keywords = ['жесть', 'трэш', 'сплетни', 'скандалы', 'депрессия', 'боль', 'ненависть', 'чп', 'дп']
hobby_keywords = ['курс', 'обучение', 'спорт', 'вязание', 'наука', 'языки', 'книги', 'творчество', 'it']

toxicity_penalties = []
hobby_bonuses = []

for u in raw_data:
    if u['id'] in users_df['user_id'].values:
        toxic_count = 0
        hobby_count = 0
        total_groups = len(u['groups'])

        for group_id, group_info in u['groups'].items():
            text_to_check = (group_info['name'] + " " + group_info['description']).lower()

            if any(word in text_to_check for word in toxic_keywords):
                toxic_count += 1
            if any(word in text_to_check for word in hobby_keywords):
                hobby_count += 1

        #доля токсичных и полезных групп, активностей и тд
        penalty = toxic_count / total_groups if total_groups > 0 else 0
        bonus = hobby_count / total_groups if total_groups > 0 else 0

        toxicity_penalties.append({'user_id': u['id'], 'toxic_penalty': penalty, 'hobby_bonus': bonus})

groups_df = pd.DataFrame(toxicity_penalties)
users_df = users_df.merge(groups_df, on='user_id', how='left')
users_df.fillna({'toxic_penalty': 0, 'hobby_bonus': 0}, inplace=True)

#Итоговая математическая формула
Из диздока: "Все показатели будут приведены к единой шкале [0, 1], а веса будут обоснованы социологическими паттернами".

In [ ]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
#применяем log1p (это натуральный логарифм от x+1, чтобы не было нуля)
users_df['friends_log'] = np.log1p(users_df['friends_num'])
users_df['activity_log'] = np.log1p(users_df['total_posts_activity'])

#нормализуем уже логарифмированные данные (от 0 до 1)
users_df['S_network'] = scaler.fit_transform(users_df[['friends_log']])
users_df['S_activity'] = scaler.fit_transform(users_df[['activity_log']])

# основа - это позитив от нейросети, плюс бонус за хобби, минус штраф за токсичность
users_df['S_sentiment'] = (users_df['dl_positive_percent'] + (users_df['hobby_bonus'] * 0.1) - (users_df['toxic_penalty'] * 0.2)).clip(0, 1)

alpha_1 = 0.25 # S_network
alpha_2 = 0.15 # S_activity
alpha_3 = 0.60 # S_sentiment

users_df['final_happiness_index'] = (users_df['S_network'] * alpha_1) + \
                                    (users_df['S_activity'] * alpha_2) + \
                                    (users_df['S_sentiment'] * alpha_3)

users_df['final_happiness_index'] = users_df['final_happiness_index'].round(3)

#Визуализация и Сегментация

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.figure(figsize=(15, 10))

#График 1: общее распределение индекса счастья
plt.subplot(2, 2, 1)
sns.histplot(users_df['final_happiness_index'], bins=20, kde=True, color="skyblue")
plt.title('Распределение Индекса цифрового счастья')
plt.xlabel('Индекс (0 до 1)')
plt.ylabel('Количество пользователей')

#График 2: Гендерный анализ
plt.subplot(2, 2, 2)
sns.boxplot(x='gender', y='final_happiness_index', data=users_df, palette="pastel")
plt.title('Уровень счастья по гендеру')
plt.xlabel('Гендер')
plt.ylabel('Индекс счастья')

#График 3: Возрастной анализ
age_df = users_df.dropna(subset=['age']).copy()
plt.subplot(2, 2, 3)
sns.regplot(x='age', y='final_happiness_index', data=age_df, scatter_kws={'alpha':0.5}, line_kws={'color':'red'})
plt.title('Зависимость счастья от возраста')
plt.xlabel('Возраст (лет)')
plt.ylabel('Индекс счастья')

plt.tight_layout()
plt.show()